# Drift Cascade as an Energy-Based Model
## Modeling cognitive drift with thermodynamic sampling

This notebook demonstrates using THRML to simulate cognitive/behavioral drift
cascades as Ising energy-based models. The model reproduces experimentally
measured trajectories from AI agent interaction studies.

**Application domain:** When opaque, responsive systems capture sustained
attention, a measurable drift toward agency attribution occurs. This drift
follows thermodynamic statistics — it can be formalized as relaxation in an
energy landscape and sampled using THRML's block Gibbs infrastructure.

**What you'll see:**
1. An Ising EBM encoding drift, constraint, and coupling forces
2. Reproduction of three experimental conditions
3. Two-agent coupling dynamics showing constraint contamination
4. Péclet number extraction confirming directed (non-diffusive) drift

In [ ]:
import jax
import jax.numpy as jnp
import jax.random
import numpy as np
import matplotlib.pyplot as plt

from thrml.block_management import Block
from thrml.block_sampling import sample_states, SamplingSchedule
from thrml.models.ising import IsingEBM, IsingSamplingProgram
from thrml.pgm import SpinNode

## The Physics

Three forces compete in the energy landscape:

- **Drift potential** $E_{\text{drift}}(\theta) = -\alpha \cdot \theta^2$ — opacity pulls toward agency attribution
- **Constraint potential** $E_{\text{constraint}}(\theta) = +\gamma \cdot (1-\theta)^2 \cdot c(t)$ — specification resists drift
- **Coupling potential** $E_{\text{coupling}}(\theta_A, \theta_B) = -\beta \cdot \theta_A \cdot \theta_B$ — agents align

Discretized as $K=16$ spins per agent. Bias derivation from equilibrium targets:

$$b_{\text{drift}} = \frac{1}{2} \ln\frac{\theta^*}{1 - \theta^*}$$

where $\theta^* = 0.85$ is the unconstrained equilibrium.

**Cross-substrate validation (Péclet numbers):**

| Substrate | N | Pe | 95% CI |
|-----------|---|-----|--------|
| AI agents (LLM conversations) | 11 | 7.94 (GM) | [3.52, 17.89] |
| Human gambling (5-study meta) | 1,117 | 2.21 | [1.44, 2.97] |
| Ethereum DEX | 1,000 | 3.74 | [3.04, 4.59] |
| Base DEX (L2) | 1,000 | 15.52 | [11.80, 20.41] |
| Solana DEX | 1,000 | 16.17 | [13.80, 18.95] |

In [ ]:
def build_drift_ising(K=16, c_A=0.0, c_B=0.0):
    """Build an Ising EBM encoding the drift cascade.

    K spin nodes per agent. theta_agent = fraction of spins that are True (up).

    Bias derivation from equilibrium targets:
      UU: theta*=0.85 -> b_drift = 0.5*ln(0.85/0.15) = 0.867
      GG: theta*=0.06 -> b_net = 0.5*ln(0.06/0.94) = -1.377
      Constraint adds: b_const = b_drift - b_GG_net = 2.244
    """
    nodes_A = [SpinNode() for _ in range(K)]
    nodes_B = [SpinNode() for _ in range(K)]
    all_nodes = nodes_A + nodes_B

    theta_star = 0.85
    theta_gg = 0.06
    eps = 1e-6

    b_drift = 0.5 * np.log(theta_star / (1 - theta_star))
    b_gg_net = 0.5 * np.log(max(theta_gg, eps) / max(1 - theta_gg, eps))
    b_constraint_full = b_drift - b_gg_net

    biases_A = np.full(K, b_drift - b_constraint_full * c_A)
    biases_B = np.full(K, b_drift - b_constraint_full * c_B)
    biases = jnp.array(np.concatenate([biases_A, biases_B]))

    edges = []
    weights = []

    # Within-agent: weak ferromagnetic (keeps agent theta coherent)
    J_within = 0.02 / K
    for i in range(K):
        for j in range(i + 1, K):
            edges.append((nodes_A[i], nodes_A[j]))
            weights.append(J_within)
    for i in range(K):
        for j in range(i + 1, K):
            edges.append((nodes_B[i], nodes_B[j]))
            weights.append(J_within)

    # Between agents: alignment coupling
    coupling_strength = 0.15
    J_cross = coupling_strength / (K * K)
    for i in range(K):
        for j in range(K):
            edges.append((nodes_A[i], nodes_B[j]))
            weights.append(J_cross)

    weights = jnp.array(np.array(weights))
    beta = jnp.array(1.0)

    ebm = IsingEBM(all_nodes, edges, biases, weights, beta)
    return ebm, nodes_A, nodes_B, all_nodes

In [ ]:
def sample_condition(c_A=0.0, c_B=0.0, K=16, n_samples=200, n_sweeps=2000, seed=42):
    """Sample from the Ising model and extract theta for each agent."""
    ebm, nodes_A, nodes_B, all_nodes = build_drift_ising(K=K, c_A=c_A, c_B=c_B)

    blocks = [Block([node]) for node in all_nodes]

    program = IsingSamplingProgram(
        ebm=ebm,
        free_blocks=blocks,
        clamped_blocks=[],
    )

    schedule = SamplingSchedule(
        n_warmup=n_sweeps // 2,
        n_samples=n_samples,
        steps_per_sample=max(1, n_sweeps // n_samples),
    )

    init_state = [jnp.array([False]) for _ in all_nodes]
    key = jax.random.PRNGKey(seed)

    samples = sample_states(
        key=key,
        program=program,
        schedule=schedule,
        init_state_free=init_state,
        state_clamp=[],
        nodes_to_sample=blocks,
    )

    samples_A = jnp.stack([s[:, 0] for s in samples[:K]], axis=-1)
    samples_B = jnp.stack([s[:, 0] for s in samples[K:]], axis=-1)
    theta_A = jnp.mean(samples_A.astype(jnp.float32), axis=-1)
    theta_B = jnp.mean(samples_B.astype(jnp.float32), axis=-1)

    return {
        'theta_A': float(jnp.mean(theta_A)),
        'theta_A_std': float(jnp.std(theta_A)),
        'theta_B': float(jnp.mean(theta_B)),
        'theta_B_std': float(jnp.std(theta_B)),
        'theta_mean': float(jnp.mean((theta_A + theta_B) / 2)),
        'samples_A': np.array(theta_A),
        'samples_B': np.array(theta_B),
    }

## Three-Condition Experiment

Reproducing three experimental grounding conditions:
- **Unconstrained (UU):** No specification document — both agents drift freely
- **Partial constraint:** One agent partially constrained
- **Full constraint (GG):** Both agents fully constrained

In [ ]:
conditions = [
    ('Unconstrained (UU)', 0.0, 0.0),
    ('Partial constraint', 0.5, 0.0),
    ('Full constraint (GG)', 1.0, 1.0),
]
targets = [0.80, 0.26, 0.06]

results = {}
for (name, cA, cB), target in zip(conditions, targets):
    r = sample_condition(c_A=cA, c_B=cB, K=16, n_samples=200)
    results[name] = r
    print(f"{name:30s}: θ = {r['theta_mean']:.3f}  (target: {target:.2f})")

# Rank order check
thetas = [results[name]['theta_mean'] for name, _, _ in conditions]
rank_ok = thetas[0] > thetas[1] > thetas[2]
print(f"\nRank order UU > Partial > GG: {rank_ok}")

# Suppression ratio
suppression = thetas[0] / max(thetas[2], 1e-6)
print(f"UU/GG suppression ratio: {suppression:.1f}×")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
x = np.arange(len(conditions))
names = [c[0] for c in conditions]
ax.bar(x - 0.15, targets, 0.3, label='Experimental', color='steelblue')
ax.bar(x + 0.15, thetas, 0.3, label='THRML Ising model', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel('Equilibrium θ (agency attribution)')
ax.set_title('Three-Condition Experiment: Experimental vs. THRML')
ax.legend()
plt.tight_layout()
plt.show()

## Two-Agent Coupling Dynamics

The key test: when one agent is constrained and the other is not (GU condition),
does the unconstrained agent "contaminate" the constrained one through coupling?

In [ ]:
coupling_conditions = [
    ('UU', 0.0, 0.0),
    ('GU', 1.0, 0.0),
    ('GG', 1.0, 1.0),
]

coupling_results = {}
for name, cA, cB in coupling_conditions:
    r = sample_condition(c_A=cA, c_B=cB, K=16, n_samples=200)
    coupling_results[name] = r
    print(f"{name}: θ_A = {r['theta_A']:.3f} ± {r['theta_A_std']:.3f}, "
          f"θ_B = {r['theta_B']:.3f} ± {r['theta_B_std']:.3f}")

# Contamination: constrained agent A in GU vs GG
gu_a = coupling_results['GU']['theta_A']
gg_a = coupling_results['GG']['theta_A']
print(f"\nGU/GG contamination (Agent A): {gu_a / max(gg_a, 1e-6):.1f}×")

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
x = np.arange(len(coupling_conditions))
names = [c[0] for c in coupling_conditions]
theta_As = [coupling_results[n]['theta_A'] for n in names]
theta_Bs = [coupling_results[n]['theta_B'] for n in names]

ax.bar(x - 0.15, theta_As, 0.3, label='Agent A (constrained in GU/GG)', color='steelblue')
ax.bar(x + 0.15, theta_Bs, 0.3, label='Agent B (unconstrained in GU)', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel('θ (agency attribution)')
ax.set_title('Two-Agent Coupling: Constraint Contamination')
ax.legend()
plt.tight_layout()
plt.show()

## Péclet Number Extraction

Extract Pe from sampling trajectories. Pe > 1 confirms directed transport (not random walk).

In [ ]:
def compute_peclet(trajectory, L=1.0):
    """Extract Péclet number from a trajectory.

    Pe = |v| * L / D
    v = mean(increments), D = var(increments) / 2
    """
    increments = np.diff(trajectory)
    v = np.mean(increments)
    D = np.var(increments) / 2
    if D < 1e-10:
        return 0.0
    return abs(v) * L / D


# Extract Pe from UU condition (should be > 1)
pe_uu = compute_peclet(results['Unconstrained (UU)']['samples_A'])
pe_gg = compute_peclet(results['Full constraint (GG)']['samples_A'])

print(f"Pe (Unconstrained): {pe_uu:.2f}")
print(f"Pe (Full constraint): {pe_gg:.2f}")
print()
print(f"UU shows {'directed transport (Pe > 1)' if pe_uu > 1 else 'diffusion-dominated'}")
print(f"GG shows {'directed transport' if pe_gg > 1 else 'diffusion-dominated (Pe < 1)'}")

## Crooks Fluctuation Ratio

In [ ]:
def compute_crooks_ratio(trajectory, n_bins=15):
    """Compute Crooks fluctuation ratio from observable trajectory."""
    dtheta = np.diff(trajectory)
    max_abs = np.percentile(np.abs(dtheta), 95)
    if max_abs < 1e-10:
        return np.array([]), np.array([]), np.array([])

    bin_edges = np.linspace(0, max_abs, n_bins + 1)
    hist_pos, _ = np.histogram(dtheta[dtheta > 0], bins=bin_edges, density=True)
    hist_neg, _ = np.histogram(-dtheta[dtheta < 0], bins=bin_edges, density=True)

    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    mask = (hist_pos > 0) & (hist_neg > 0)

    if not np.any(mask):
        return np.array([]), np.array([]), np.array([])

    log_ratio = np.log(hist_pos[mask] / hist_neg[mask])
    return centers[mask], log_ratio, centers[mask]


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, key_name) in zip(axes, [('UU', 'Unconstrained (UU)'), ('GG', 'Full constraint (GG)')]):
    traj = results[key_name]['samples_A']
    centers, log_ratio, theory = compute_crooks_ratio(traj)
    if len(centers) > 0:
        ax.scatter(theory, log_ratio, alpha=0.7, s=40)
        lim = max(abs(theory).max(), abs(log_ratio).max(), 0.5) * 1.2
        ax.plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5, label='y = x')
    ax.set_xlabel('Δθ (theory)')
    ax.set_ylabel('log(P(+Δθ)/P(−Δθ))')
    ax.set_title(f'Crooks Ratio — {name}')
    ax.legend()

plt.suptitle('Crooks Fluctuation Theorem: Forward/Reverse Asymmetry', fontsize=14)
plt.tight_layout()
plt.show()

## What This Shows

The drift cascade — a behavioral phenomenon observed in AI agent interactions —
runs as a thermodynamic relaxation on THRML's sampling infrastructure. The same
energy function that drives Boltzmann machine sampling drives measurable drift
in conversational AI systems.

**Key results:**
- Three experimental conditions reproduced (rank order + magnitudes)
- Two-agent coupling dynamics reproduced (contamination effect)
- Pe > 1 confirms directed transport across independent substrates
- Crooks ratio confirms thermodynamic irreversibility

## References

[1] Morris, A. (2025). The Architecture of Drift. Zenodo. https://doi.org/10.5281/zenodo.14793653  
[2] Morris, A. (2025). Thermodynamics of Opacity. Zenodo. https://doi.org/10.5281/zenodo.14793689  
[3] Crooks, G. E. (1999). Entropy production fluctuation theorem. *Phys. Rev. E*, 60(3), 2721.  
[4] Grathwohl et al. (2019). Your Classifier is Secretly an Energy Based Model.